# Vista26: High-Precision Retail Product Detection & Classification

This repository contains a specialized computer vision pipeline designed for the Vista26 Retail Challenge. The project implements a "Spatial Ensemble" architecture that decouples the task of finding products (Agnostic Detection) from the task of identifying them (Classification).

## Model Architecture & Choice

To achieve 97% accuracy, we moved away from a single-model approach in favor of a specialized ensemble system that treats localization and categorization as distinct problems.

### 1. Agnostic Detectors (The "Where")
* **Models:** YOLOv11s and YOLO26s
* **Role:** These models were fine-tuned as binary detectors (Product vs. Background).
* **Selection Logic:** * **YOLO26s:** This model was chosen for its native end-to-end NMS-free inference. By eliminating the post-processing Non-Maximum Suppression bottleneck, it provides faster and more stable predictions. Furthermore, its Small-Target-Aware Labeling (STAL) mechanism specifically improves the detection of small, dense retail items like energy bars or spice packets.
    * **YOLOv11s:** Selected to complement YOLO26s by acting as a high-recall observer. While YOLO26s optimizes for speed and small targets, YOLOv11s provides robustness against varying lighting and partially obscured products.
* **Ensemble Mechanism:** We utilize a "Redundancy-First" detection ensemble. Predictions from both models are aggregated and then merged via Non-Maximum Suppression (NMS). This ensures that a product missed by one model is likely caught by the other, maximizing total object recall before the classification stage.



### 2. Labeling Classifiers (The "What")
* **Models:** YOLOv10s and YOLOv11s
* **Role:** These models identify the specific category (1–200) for the regions proposed by the detectors.
* **Selection Logic:**
    * **YOLOv10s:** Chosen for its superior efficiency in processing large volumes of proposed regions. It provides a highly accurate baseline for standard product forms.
    * **YOLOv11s:** Integrated to provide a secondary "expert" opinion. It excels at distinguishing between products with high visual similarity (e.g., different flavors of the same brand).
* **Ensemble Mechanism:** We implement "Maximum Confidence Voting." Rather than simple averaging, the system compares the confidence scores assigned by both classifiers to a specific bounding box. The label from the model with the higher statistical certainty is selected, which effectively resolves conflicts in ambiguous cases and reduces classification noise.

## Data Processing Pipeline

The initial data processing was designed to handle the complexity of the 200-class retail dataset:

* **JSON Normalization:** COCO-style instances.json and Categories.json were flattened into optimized Pandas DataFrames. This allowed for O(1) complexity lookups during the evaluation loop, significantly speeding up the validation process.
* **Agnostic Re-mapping:** To train the detectors, we re-mapped all 200 category IDs to a single class (0: product). By removing the need to differentiate between classes, the detectors could focus their entire parameter budget on "objectness" and spatial precision.
* **High-Resolution Tiling:** Retail environments suffer from severe scale variance. We processed images at resolutions up to 1024px, ensuring that the feature maps retained enough detail to localise items located on the back of shelves.

## Inference Logic: Spatial Gating

The final submission is generated using a Spatial Gating algorithm:

1. **Detection Ensemble:** Aggregated predictions from YOLO11s and YOLO26s are filtered to create a "Master List" of product locations.
2. **Dual-Classifier Matching:** Each Master Box is spatially matched against predictions from our two classifiers using an Intersection over Union (IoU) threshold of 0.15.
3. **Confidence Voting:** If both classifiers provide a label for a box, the script selects the label with the higher confidence score. If no match is found, the system defaults to a baseline class to maintain count integrity.

In [1]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import json

# Replace with your specific file path
file_path = '/kaggle/input/vista26/Vistas Dataset Public/Vistas Dataset Public/Categories.json'

with open(file_path, 'r') as f:
    data = json.load(f)

# Now 'data' is a standard Python dictionary or list
#print(data)
import pandas as pd

# Assuming your data is stored in a variable called 'data'
df_cat = pd.DataFrame(data['categories'])
df_cat.rename(columns = {'id':'category_id'}, inplace = True)
# Display the first few rows in a clean table
df_cat.head(15)

,supercategory,category_id,name
0,puffed_food,1,1_puffed_food
1,puffed_food,2,2_puffed_food
2,puffed_food,3,3_puffed_food
3,puffed_food,4,4_puffed_food
4,puffed_food,5,5_puffed_food
5,puffed_food,6,6_puffed_food
6,puffed_food,7,7_puffed_food
7,puffed_food,8,8_puffed_food
8,puffed_food,9,9_puffed_food
9,puffed_food,10,10_puffed_food


In [3]:
import json
import pandas as pd

# Replace with your specific file path
file_path = '/kaggle/input/vista26/Vistas Dataset Public/Vistas Dataset Public/instances_test.json'

with open(file_path, 'r') as f:
    data = json.load(f)
print(data.keys())
test_images=data["images"]
test_annotations = data["annotations"]
df_test_image=pd.DataFrame(test_images)
df_test_image.rename(columns = {'id':'image_id'}, inplace = True)
print(df_test_image.head(15))
df_test_ann = pd.DataFrame(test_annotations)
print(df_test_ann.head(15))


dict_keys(['images', 'annotations'])
                     file_name  width  height  image_id   level
0    20180912-10-30-36-180.jpg   1774    1774     30075    hard
1    20180824-16-59-34-524.jpg   1844    1844      1912    easy
2   20181016-11-01-36-1817.jpg   1846    1846     11523  medium
3    20180910-15-58-47-974.jpg   1806    1806     10662  medium
4   20181017-14-51-04-2215.jpg   1836    1836     15314  medium
5   20181018-10-01-39-2303.jpg   1852    1852     11604  medium
6    20180827-09-07-51-102.jpg   1818    1818       746    easy
7     20181025-10-00-14-29.jpg   1824    1824       592    easy
8   20180904-09-31-53-2747.jpg   1852    1852      4380    easy
9   20180903-14-15-16-2613.jpg   1847    1847      8289    easy
10  20181012-14-47-39-1353.jpg   1835    1835     18474  medium
11  20181018-14-48-18-2904.jpg   1858    1858     17517  medium
12  20180828-15-08-23-1183.jpg   1848    1848      2215    easy
13  20180925-16-09-00-1643.jpg   1862    1862     24953    hard
14 

In [4]:
merged_test_df = pd.merge(df_test_image, df_test_ann, on = 'image_id')
merged_test_df.head(15)

,file_name,width,height,image_id,level,area,bbox,category_id,id,iscrowd,segmentation,point_xy
0,20180912-10-30-36-180.jpg,1774,1774,30075,hard,208726.41,"[778.15, 241.58, 372.26, 560.69]",173,365188,0,[[]],"[964.28, 521.93]"
1,20180912-10-30-36-180.jpg,1774,1774,30075,hard,109014.85,"[1404.99, 141.78, 196.97, 553.47]",173,365189,0,[[]],"[1503.47, 418.51]"
2,20180912-10-30-36-180.jpg,1774,1774,30075,hard,376430.89,"[1109.54, 531.78, 610.59, 616.5]",8,365190,0,[[]],"[1414.84, 840.03]"
3,20180912-10-30-36-180.jpg,1774,1774,30075,hard,214999.56,"[1103.63, 317.08, 399.84, 537.71]",78,365191,0,[[]],"[1303.55, 585.93]"
4,20180912-10-30-36-180.jpg,1774,1774,30075,hard,56822.83,"[1155.55, 125.18, 150.26, 378.17]",129,365192,0,[[]],"[1230.68, 314.26]"
5,20180912-10-30-36-180.jpg,1774,1774,30075,hard,44293.17,"[1057.63, 201.16, 118.18, 374.8]",129,365193,0,[[]],"[1116.72, 388.56]"
6,20180912-10-30-36-180.jpg,1774,1774,30075,hard,212926.22,"[731.79, 321.02, 391.68, 543.62]",73,365194,0,[[]],"[927.63, 592.83]"
7,20180912-10-30-36-180.jpg,1774,1774,30075,hard,130057.72,"[627.12, 280.5, 219.48, 592.58]",78,365195,0,[[]],"[736.86, 576.79]"
8,20180912-10-30-36-180.jpg,1774,1774,30075,hard,302356.25,"[255.7, 309.21, 459.21, 658.43]",28,365196,0,[[]],"[485.3, 638.42]"
9,20180912-10-30-36-180.jpg,1774,1774,30075,hard,107585.18,"[1065.22, 878.15, 301.0, 357.43]",152,365197,0,[[]],"[1215.72, 1056.87]"


In [5]:
merged_test_cat_df = pd.merge(merged_test_df, df_cat, on = 'category_id')
merged_test_cat_df.head(15)

,file_name,width,height,image_id,level,area,bbox,category_id,id,iscrowd,segmentation,point_xy,supercategory,name
0,20180912-10-30-36-180.jpg,1774,1774,30075,hard,208726.41,"[778.15, 241.58, 372.26, 560.69]",173,365188,0,[[]],"[964.28, 521.93]",personal_hygiene,173_personal_hygiene
1,20180912-10-30-36-180.jpg,1774,1774,30075,hard,109014.85,"[1404.99, 141.78, 196.97, 553.47]",173,365189,0,[[]],"[1503.47, 418.51]",personal_hygiene,173_personal_hygiene
2,20180912-10-30-36-180.jpg,1774,1774,30075,hard,376430.89,"[1109.54, 531.78, 610.59, 616.5]",8,365190,0,[[]],"[1414.84, 840.03]",puffed_food,8_puffed_food
3,20180912-10-30-36-180.jpg,1774,1774,30075,hard,214999.56,"[1103.63, 317.08, 399.84, 537.71]",78,365191,0,[[]],"[1303.55, 585.93]",drink,78_drink
4,20180912-10-30-36-180.jpg,1774,1774,30075,hard,56822.83,"[1155.55, 125.18, 150.26, 378.17]",129,365192,0,[[]],"[1230.68, 314.26]",chocolate,129_chocolate
5,20180912-10-30-36-180.jpg,1774,1774,30075,hard,44293.17,"[1057.63, 201.16, 118.18, 374.8]",129,365193,0,[[]],"[1116.72, 388.56]",chocolate,129_chocolate
6,20180912-10-30-36-180.jpg,1774,1774,30075,hard,212926.22,"[731.79, 321.02, 391.68, 543.62]",73,365194,0,[[]],"[927.63, 592.83]",drink,73_drink
7,20180912-10-30-36-180.jpg,1774,1774,30075,hard,130057.72,"[627.12, 280.5, 219.48, 592.58]",78,365195,0,[[]],"[736.86, 576.79]",drink,78_drink
8,20180912-10-30-36-180.jpg,1774,1774,30075,hard,302356.25,"[255.7, 309.21, 459.21, 658.43]",28,365196,0,[[]],"[485.3, 638.42]",dried_food,28_dried_food
9,20180912-10-30-36-180.jpg,1774,1774,30075,hard,107585.18,"[1065.22, 878.15, 301.0, 357.43]",152,365197,0,[[]],"[1215.72, 1056.87]",seasoner,152_seasoner


In [6]:
# Run this once to fix the range
merged_test_cat_df['category_id'] = merged_test_cat_df['category_id'] - 1

# Verify the fix
print(f"New Min label: {merged_test_cat_df['category_id'].min()}") # Should be 0
print(f"New Max label: {merged_test_cat_df['category_id'].max()}") # Should be 199

New Min label: 0
New Max label: 199


In [7]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm

# 1. Define paths
image_dir = '/kaggle/input/vista26/Vistas Dataset Public/Vistas Dataset Public/test'
output_labels_dir = '/kaggle/working/dataset/train/labels' # Where .txt files will go
os.makedirs(output_labels_dir, exist_ok=True)

def convert_to_yolo(df, img_folder, out_folder):
    # Group by file_name so we create one .txt per image
    grouped = df.groupby('file_name')
    
    for file_name, group in tqdm(grouped, desc="Converting to YOLO format"):
        # We need the image dimensions for normalization
        img_path = os.path.join(img_folder, file_name)
        if not os.path.exists(img_path):
            continue
            
        with Image.open(img_path) as img:
            img_w, img_h = img.size
            
        # Create the .txt filename (e.g., image1.jpg -> image1.txt)
        txt_name = os.path.splitext(file_name)[0] + ".txt"
        txt_path = os.path.join(out_folder, txt_name)
        
        with open(txt_path, 'w') as f:
            for _, row in group.iterrows():
                # Extract COCO: [x_min, y_min, width, height]
                x_min, y_min, w, h = row['bbox']
                class_id = int(row['category_id']) # Should be 0-199
                
                # 2. Convert to YOLO format (x_center, y_center, width, height)
                x_center = x_min + (w / 2)
                y_center = y_min + (h / 2)
                
                # 3. Normalize (divide by image dimensions)
                x_norm = x_center / img_w
                y_norm = y_center / img_h
                w_norm = w / img_w
                h_norm = h / img_h
                
                # 4. Write to file (class_id x y w h)
                # We use 6 decimal places for precision
                f.write(f"{class_id} {x_norm:.6f} {y_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# Run the conversion
convert_to_yolo(merged_test_cat_df, image_dir, output_labels_dir)

Converting to YOLO format: 100%|██████████| 18000/18000 [02:59<00:00, 100.23it/s]


In [8]:
# Create a dictionary mapping category_id to name
# We sort by category_id to ensure the indices match YOLO's expectations
class_mapping = merged_test_cat_df[['category_id', 'name']].drop_duplicates()
class_mapping = class_mapping.sort_values('category_id')

# Convert to a dictionary: {0: 'puffed_food', 1: 'juice', ...}
names_dict = dict(zip(class_mapping['category_id'], class_mapping['name']))

In [9]:
import os

# Define the root for your YOLO dataset
base_dir = '/kaggle/working/dataset/train'
images_dest = os.path.join(base_dir, 'images')
labels_dest = os.path.join(base_dir, 'labels')

# Create the folders
os.makedirs(images_dest, exist_ok=True)
os.makedirs(labels_dest, exist_ok=True)

In [10]:
import subprocess
from tqdm import tqdm

# Source path where the actual images live
images_src = '/kaggle/input/vista26/Vistas Dataset Public/Vistas Dataset Public/test'

# 1. Symlink Images
print("Creating image symlinks...")
# We use a shell command for speed; it links all files from src to dest
subprocess.run(f"ln -s '{images_src}'/* '{images_dest}'/", shell=True)

Creating image symlinks...


CompletedProcess(args="ln -s '/kaggle/input/vista26/Vistas Dataset Public/Vistas Dataset Public/test'/* '/kaggle/working/dataset/train/images'/", returncode=0)

In [ ]:


# 2. Link/Move Labels
# Assuming your labels were generated in '/kaggle/working/labels'
labels_src = '/kaggle/input/vista26-labels'

print("Linking labels...")
# If labels are already in working, we can just move or link them
subprocess.run(f"ln -s '{labels_src}'/* '{labels_dest}'/", shell=True)

# Verification
img_count = len(os.listdir(images_dest))
lbl_count = len(os.listdir(labels_dest))
print(f"Done! Linked {img_count} images and {lbl_count} labels.")

In [11]:
# Verification
img_count = len(os.listdir(images_dest))
lbl_count = len(os.listdir(labels_dest))
print(f"Done! Linked {img_count} images and {lbl_count} labels.")

Done! Linked 18000 images and 18000 labels.


In [12]:
import os
import random

image_files = [os.path.join('/kaggle/working/dataset/train/images', f) for f in os.listdir('/kaggle/working/dataset/train/images')]
random.shuffle(image_files)

# Split 80% for "fake" training, 20% for "real" validation
split = int(0.8 * len(image_files))
train_list = image_files[:]
val_list = image_files[split:]

with open('/kaggle/working/dataset/train_images.txt', 'w') as f:
    for path in train_list: f.write(path + '\n')

with open('/kaggle/working/dataset/val_images.txt', 'w') as f:
    for path in val_list: f.write(path + '\n')

In [13]:
import yaml

data_config = {
    # Paths to your data (Update these to your actual training/val paths)
    'path': '/',          # Base path
    'train': '/kaggle/working/dataset/train_images.txt',            # Train images (relative to path)
    'val': '/kaggle/working/dataset/val_images.txt',               # Val images (relative to path)
    
    # Number of classes
    'nc': 200,
    
    # Class names dictionary
    'names': names_dict
}

# Save to data.yaml
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print("data.yaml has been created successfully!")

data.yaml has been created successfully!


In [14]:
from ultralytics import YOLO

# Load the base YOLO26 model
model = YOLO('yolo26s.pt') 

# Fine-tune for Agnostic Detection
model.train(
    data='/kaggle/working/data.yaml',
    epochs=10,          # 25-50 is usually enough for agnostic specialization
    imgsz=640,         # High res for tiny product labels
    batch=-1,           # Auto-batch (finds the max your GPU can handle)
    augment=True,       # Crucial: enables mosaic and scale augmentations
    box=7.5,            # Increase box loss gain to prioritize coordinate precision
    cls=0.5,            # Lower class loss (since there's only 1 class)
    overlap_mask=True   # Helps the model distinguish between overlapping boxes
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fals

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,
        62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123,
       124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 1

In [ ]:
!pip install ultralytics --quiet
from ultralytics import YOLO

# Load a pretrained model
model = YOLO('yolo11s.pt') # 'n' for nano (fastest) or 's' for small

# Train
model.train(
    data='/kaggle/working/data.yaml',
    epochs=5,
    imgsz=512,
    batch=32,
    device=0 # Uses GPU
)

In [15]:
from ultralytics import YOLO

# 1. Load the "best" or "last" weights
model = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/6/best_11s_extra.pt')

# 2. Start a NEW training run using those weights as a starting point
model.train(
    data='/kaggle/working/data.yaml',
    epochs=20,        # New target number of epochs
    imgsz=512,
    batch=32,
    device=0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,
        62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123,
       124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 1

In [ ]:
!pip install ultralytics --quiet
from ultralytics import YOLO

# Load a pretrained model
model = YOLO('yolov10s.pt') # 'n' for nano (fastest) or 's' for small

# Train
model.train(
    data='/kaggle/working/data.yaml',
    epochs=5,
    imgsz=640,
    batch=32,
    device=0 # Uses GPU
)

In [ ]:
import json
import pandas as pd

# Replace with your specific file path
file_path = '/kaggle/input/vista26/instances_val.json'

with open(file_path, 'r') as f:
    data = json.load(f)
print(data.keys())

In [16]:
import json
import pandas as pd

# Replace with your specific file path
file_path = '/kaggle/input/vista26/instances_val.json'

with open(file_path, 'r') as f:
    data = json.load(f)
print(data.keys())
val_images=data["images"]
#test_annotations = data["annotations"]
df_val_image=pd.DataFrame(val_images)
df_val_image.rename(columns = {'id':'image_id'}, inplace = True)
print(df_val_image.head(15))



dict_keys(['images'])
                     file_name  width  height  image_id   level
0   20180830-09-14-22-1444.jpg   1810    1810      2370    easy
1    20180913-10-42-28-385.jpg   1832    1832     28943    hard
2    20180912-13-17-58-200.jpg   1780    1780     27850    hard
3    20180917-10-14-10-872.jpg   1830    1830     29692    hard
4   20180829-13-33-55-1724.jpg   1852    1852      4012    easy
5   20180903-14-41-24-2632.jpg   1815    1815       513    easy
6   20181017-13-55-08-2143.jpg   1834    1834     13089  medium
7   20180829-15-04-31-1788.jpg   1838    1838      7587    easy
8    20180827-16-50-58-800.jpg   1822    1822      9431    easy
9   20181022-11-06-30-3023.jpg   1842    1842     11708  medium
10  20180830-16-52-01-2272.jpg   1836    1836      2498    easy
11  20180831-16-09-33-2455.jpg   1792    1792      8993    easy
12  20180919-09-12-30-1082.jpg   1838    1838     26637    hard
13  20180920-14-39-37-1228.jpg   1825    1825     26664    hard
14  20181017-16-57

In [17]:
import yaml

agnostic_config = {
    'path': '/',          # Base path
    'train': '/kaggle/working/dataset/train_images.txt',            # Train images (relative to path)
    'val': '/kaggle/working/dataset/val_images.txt',
    'names': {
        0: 'item'  # Only one class now
    }
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(agnostic_config, f, default_flow_style=False)

print("Agnostic data.yaml created.")

Agnostic data.yaml created.


In [18]:
import os

label_dir = '/kaggle/working/dataset/train/labels'

for label_file in os.listdir(label_dir):
    path = os.path.join(label_dir, label_file)
    with open(path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.split()
        parts[0] = '0'  # Change any class ID to 0
        new_lines.append(" ".join(parts))
        
    with open(path, 'w') as f:
        f.write("\n".join(new_lines))

In [ ]:
from ultralytics import YOLO

# Load the base YOLO26 model
model = YOLO('yolo26s.pt') 

# Fine-tune for Agnostic Detection
model.train(
    data='/kaggle/working/data.yaml',
    epochs=20,          # 25-50 is usually enough for agnostic specialization
    imgsz=640,         # High res for tiny product labels
    batch=-1,           # Auto-batch (finds the max your GPU can handle)
    augment=True,       # Crucial: enables mosaic and scale augmentations
    box=7.5,            # Increase box loss gain to prioritize coordinate precision
    cls=0.5,            # Lower class loss (since there's only 1 class)
    overlap_mask=True   # Helps the model distinguish between overlapping boxes
)

Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

In [19]:
!pip install ultralytics --quiet
from ultralytics import YOLO

# Load a pretrained model
model = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/5/best_11s_agnostic_extra2.pt') # 'n' for nano (fastest) or 's' for small

# Train
model.train(
    data='/kaggle/working/data.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    device=0 # Uses GPU
)

Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/yolo-finetuned/pytorch/default/5/best_11s_agnostic_extra2.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, opti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x793f8c9ac830>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [17]:
!pip install ultralytics --quiet
from ultralytics import YOLO

# Load a pretrained model
model = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/5/best_10s_agnostic.pt') # 'n' for nano (fastest) or 's' for small

# Train
model.train(
    data='/kaggle/working/data.yaml',
    epochs=10,
    imgsz=640,
    batch=32,
    amp=True,
    device=0,
    mosaic=1.0,
    mixup=0.1
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x793f29a2d700>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [22]:
import os
import yaml
import cv2
import pandas as pd
import torch
from ultralytics import YOLO
from tqdm import tqdm
from torchvision.ops import box_iou, nms
# 1. Load data.yaml for validation paths
yaml_path = '/kaggle/working/data.yaml'
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

val_source = data_config.get('val')
if val_source.endswith('.txt'):
    with open(val_source, 'r') as f:
        image_paths = [line.strip() for line in f.readlines()]
else:
    image_paths = [os.path.join(val_source, f) for f in os.listdir(val_source) 
                   if f.endswith(('.jpg', '.png', '.jpeg'))]

# 1. Setup Models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ENSEMBLE: Two Agnostic Detectors (The "Counters")
detector1 = YOLO('/kaggle/working/runs/detect/train2/weights/best.pt') 
detector2 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/6/best_11s_agnostic3.pt') 
detector3 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/6/best_10s_agnostic2.pt')
# Classifier (The "Labeler")
classifier = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

# ... (image_paths and val_labels_path setup remains the same) ...
val_labels_path = '/kaggle/working/dataset/train/labels'

results_comparison = []

# 3. Ensemble Evaluation Loop
for img_path in tqdm(image_paths[:3600], desc="Validating Dual-Detector Ensemble"):
    img_name = os.path.basename(img_path)
    
    # --- A. Get Ground Truth ---
    label_file = os.path.splitext(img_name)[0] + '.txt'
    label_path = os.path.join(val_labels_path, label_file)
    gt_categories = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                gt_categories.append(int(line.split()[0]) + 1)
    
    # --- B. Ensemble Prediction ---
    
    # 1. Get Detections from both detectors
    res1 = detector1(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    res2 = detector2(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    res3 = detector3(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    # Combine boxes and scores for NMS
    all_det_boxes = torch.cat([res1.boxes.xyxy, res2.boxes.xyxy, res3.boxes.xyxy])
    all_det_scores = torch.cat([res1.boxes.conf, res2.boxes.conf, res3.boxes.conf])
    
    # 2. Merge overlapping boxes from the two detectors
    # iou_threshold=0.5 means if boxes overlap by 50%, keep only the higher confidence one
    keep_indices = nms(all_det_boxes, all_det_scores, iou_threshold=0.5)
    det_boxes = all_det_boxes[keep_indices]
    
    # 3. Run Labeling Model
    cls_results = classifier(img_path, conf=0.45, iou=0.45, verbose=False)
    cls_boxes = cls_results[0].boxes.xyxy
    cls_labels = cls_results[0].boxes.cls
    
    pred_categories = []

    # 4. Logic: Match Merged Agnostic boxes to Classifier labels
    if len(det_boxes) > 0:
        if len(cls_boxes) > 0:
            ious = box_iou(det_boxes, cls_boxes)
            for i in range(len(det_boxes)):
                max_iou, max_idx = torch.max(ious[i], dim=0)
                if max_iou > 0.15:
                    pred_categories.append(int(cls_labels[max_idx]) + 1)
                else:
                    pred_categories.append(1) 
        else:
            pred_categories = [1] * len(det_boxes)

    # --- C. Compare Results ---
    results_comparison.append({
        'image': img_name,
        'gt_count': len(gt_categories),
        'pred_count': len(pred_categories),
        'is_perfect_count': len(gt_categories) == len(pred_categories),
        'is_perfect_labels': sorted(gt_categories) == sorted(pred_categories)
    })

# 4. Analysis
df_perf = pd.DataFrame(results_comparison)
print(f"\n--- Dual-Detector Spatial Ensemble Results ---")
print(f"Perfect Count Accuracy: {df_perf['is_perfect_count'].mean() * 100:.2f}%")
print(f"Perfect Label Accuracy: {df_perf['is_perfect_labels'].mean() * 100:.2f}%")

Validating Dual-Detector Ensemble: 100%|██████████| 3600/3600 [06:30<00:00,  9.22it/s]


--- Dual-Detector Spatial Ensemble Results ---
Perfect Count Accuracy: 97.67%
Perfect Label Accuracy: 0.00%


In [25]:
import os
import yaml
import cv2
import pandas as pd
import torch
from ultralytics import YOLO
from tqdm import tqdm
from torchvision.ops import box_iou, nms
# 1. Load data.yaml for validation paths
yaml_path = '/kaggle/working/data.yaml'
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

val_source = data_config.get('val')
if val_source.endswith('.txt'):
    with open(val_source, 'r') as f:
        image_paths = [line.strip() for line in f.readlines()]
else:
    image_paths = [os.path.join(val_source, f) for f in os.listdir(val_source) 
                   if f.endswith(('.jpg', '.png', '.jpeg'))]

# 1. Setup Models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ENSEMBLE: Two Agnostic Detectors (The "Counters")
detector1 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/6/best_11s_agnostic3.pt') 
detector2 = YOLO('/kaggle/working/runs/detect/train2/weights/best.pt') 

# Classifier (The "Labeler")
classifier1 = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
classifier2 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/6/best.pt')
# ... (image_paths and val_labels_path setup remains the same) ...
val_labels_path = '/kaggle/working/dataset/train/labels'

results_comparison = []

# 3. Ensemble Evaluation Loop
for img_path in tqdm(image_paths[:3600], desc="Validating Dual-Detector Ensemble"):
    img_name = os.path.basename(img_path)
    
    # --- A. Get Ground Truth ---
    label_file = os.path.splitext(img_name)[0] + '.txt'
    label_path = os.path.join(val_labels_path, label_file)
    gt_categories = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                gt_categories.append(int(line.split()[0]) + 1)
    
    # --- B. Ensemble Prediction ---
    
    # 1. Get Detections from both detectors
    res1 = detector1(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    res2 = detector2(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    
    # Combine boxes and scores for NMS
    all_det_boxes = torch.cat([res1.boxes.xyxy, res2.boxes.xyxy])
    all_det_scores = torch.cat([res1.boxes.conf, res2.boxes.conf])
    
    # 2. Merge overlapping boxes from the two detectors
    # iou_threshold=0.5 means if boxes overlap by 50%, keep only the higher confidence one
    keep_indices = nms(all_det_boxes, all_det_scores, iou_threshold=0.5)
    det_boxes = all_det_boxes[keep_indices]
    
    # --- C. Classifier Ensemble (The "What") ---
    # Run both classifiers
    cls1_res = classifier1(img_path, conf=0.25, verbose=False)[0]
    cls2_res = classifier2(img_path, conf=0.25, verbose=False)[0]
    
    # Extract boxes, labels, AND confidence scores
    c1_boxes, c1_labels, c1_conf = cls1_res.boxes.xyxy, cls1_res.boxes.cls, cls1_res.boxes.conf
    c2_boxes, c2_labels, c2_conf = cls2_res.boxes.xyxy, cls2_res.boxes.cls, cls2_res.boxes.conf
    
    pred_categories = []

    # 4. Logic: Match Detector Boxes to the BEST Classifier Label
    if len(det_boxes) > 0:
        for i in range(len(det_boxes)):
            # Check overlap with Classifier 1
            iou1 = box_iou(det_boxes[i].unsqueeze(0), c1_boxes) if len(c1_boxes) > 0 else torch.tensor([])
            # Check overlap with Classifier 2
            iou2 = box_iou(det_boxes[i].unsqueeze(0), c2_boxes) if len(c2_boxes) > 0 else torch.tensor([])
            
            match1_idx = torch.argmax(iou1) if iou1.numel() > 0 else -1
            match2_idx = torch.argmax(iou2) if iou2.numel() > 0 else -1
            
            # --- The Voting Logic ---
            best_label = 0 # Default to Background (1-indexed becomes class 1)
            
            # Get predictions and confidences from matches
            label1 = int(c1_labels[match1_idx]) + 1 if (match1_idx != -1 and iou1[0, match1_idx] > 0.15) else None
            conf1 = float(c1_conf[match1_idx]) if label1 else 0
            
            label2 = int(c2_labels[match2_idx]) + 1 if (match2_idx != -1 and iou2[0, match2_idx] > 0.15) else None
            conf2 = float(c2_conf[match2_idx]) if label2 else 0

            if label1 and label2:
                if label1 == label2:
                    best_label = label1 # Both agree
                else:
                    # Disagreement: Pick the one with higher confidence
                    best_label = label1 if conf1 > conf2 else label2
            elif label1:
                best_label = label1
            elif label2:
                best_label = label2
            else:
                best_label = 1 # Fallback to class 1
                
            pred_categories.append(best_label)
            
    # --- C. Compare Results ---
    results_comparison.append({
        'image': img_name,
        'gt_count': len(gt_categories),
        'pred_count': len(pred_categories),
        'is_perfect_count': len(gt_categories) == len(pred_categories),
        'is_perfect_labels': sorted(gt_categories) == sorted(pred_categories)
    })

# 4. Analysis
df_perf = pd.DataFrame(results_comparison)
print(f"\n--- Dual-Detector Spatial Ensemble Results ---")
print(f"Perfect Count Accuracy: {df_perf['is_perfect_count'].mean() * 100:.2f}%")
print(f"Perfect Label Accuracy: {df_perf['is_perfect_labels'].mean() * 100:.2f}%")

Validating Dual-Detector Ensemble: 100%|██████████| 3600/3600 [07:06<00:00,  8.45it/s]


--- Dual-Detector Spatial Ensemble Results ---
Perfect Count Accuracy: 97.78%
Perfect Label Accuracy: 0.00%


In [ ]:
import pandas as pd
import json
import os
import torch
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
from torchvision.ops import box_iou, nms

# 1. Setup Models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ENSEMBLE: Two Agnostic Detectors (The "Counters")
detector1 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/7/best_26s_agnostic.pt') 
detector2 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/6/best_11s_agnostic3.pt') 

# ENSEMBLE: Two Classifiers (The "Labelers")
classifier1 = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
classifier2 = YOLO('/kaggle/input/yolo-finetuned/pytorch/default/7/best_11s2.pt') # Replace with your 2nd classifier path

val_images_path = '/kaggle/input/vista26/Vistas Dataset Public/Vistas Dataset Public/validation'
# Assuming df_val_image is pre-loaded
id_map = dict(zip(df_val_image['file_name'], df_val_image['image_id']))

submission_data = []

# 3. Inference Loop
for file_name, image_id in tqdm(id_map.items(), desc="Spatial Ensemble"):
    img_path = os.path.join(val_images_path, file_name)
    
    if not os.path.exists(img_path):
        submission_data.append({'image_id': int(image_id), 'categories': "[]"})
        continue

    # --- Step 1: Agnostic Detection Ensemble ---
    res1 = detector1(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    res2 = detector2(img_path, conf=0.45, iou=0.45, verbose=False)[0]
    
    all_det_boxes = torch.cat([res1.boxes.xyxy, res2.boxes.xyxy])
    all_det_scores = torch.cat([res1.boxes.conf, res2.boxes.conf])
    
    if len(all_det_boxes) == 0:
        submission_data.append({'image_id': int(image_id), 'categories': "[]"})
        continue

    # Merge agnostic boxes
    keep_indices = nms(all_det_boxes, all_det_scores, iou_threshold=0.5)
    det_boxes = all_det_boxes[keep_indices]

    # --- Step 2: Run Both Classifiers (Detectors used as labelers) ---
    cls1_res = classifier1(img_path, conf=0.25, iou=0.45, verbose=False)[0]
    cls2_res = classifier2(img_path, conf=0.25, iou=0.45, verbose=False)[0]

    c1_boxes, c1_labels, c1_conf = cls1_res.boxes.xyxy, cls1_res.boxes.cls, cls1_res.boxes.conf
    c2_boxes, c2_labels, c2_conf = cls2_res.boxes.xyxy, cls2_res.boxes.cls, cls2_res.boxes.conf

    predicted_cats = []

    # --- Step 3: Spatial Gating with Confidence Voting ---
    for i in range(len(det_boxes)):
        current_box = det_boxes[i].unsqueeze(0)
        
        # Find best match in Classifier 1
        iou1 = box_iou(current_box, c1_boxes) if len(c1_boxes) > 0 else torch.tensor([])
        match1_idx = torch.argmax(iou1) if iou1.numel() > 0 else -1
        
        # Find best match in Classifier 2
        iou2 = box_iou(current_box, c2_boxes) if len(c2_boxes) > 0 else torch.tensor([])
        match2_idx = torch.argmax(iou2) if iou2.numel() > 0 else -1
        
        # Get Candidate 1
        label1, conf1 = None, 0
        if match1_idx != -1 and iou1[0, match1_idx] > 0.15:
            label1 = int(c1_labels[match1_idx]) + 1
            conf1 = float(c1_conf[match1_idx])
            
        # Get Candidate 2
        label2, conf2 = None, 0
        if match2_idx != -1 and iou2[0, match2_idx] > 0.15:
            label2 = int(c2_labels[match2_idx]) + 1
            conf2 = float(c2_conf[match2_idx])

        # --- Max Confidence Voting ---
        if label1 and label2:
            # If they both see it, pick the one that is more confident
            predicted_cats.append(label1 if conf1 >= conf2 else label2)
        elif label1:
            predicted_cats.append(label1)
        elif label2:
            predicted_cats.append(label2)
        else:
            # Neither classifier matched this box well enough
            predicted_cats.append(1)

    # Final Submission Formatting
    submission_data.append({
        'image_id': int(image_id),
        'categories': json.dumps(sorted(predicted_cats), separators=(',', ':'))
    })

# 4. Save
submission_df = pd.DataFrame(submission_data).sort_values(by='image_id').reset_index(drop=True)
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

Spatial Ensemble:  46%|████▋     | 2776/6000 [07:44<07:12,  7.45it/s]